In [ ]:
# 00 — Project Setup and Data Access Check

**Project:** Short-term forecasting of AI data center power demand — a fair benchmark of classical, foundation, and hybrid quantum-classical models, with uncertainty quantification and grid-aware control.

**Data:** MIT Supercloud Dataset (GPU power time series, 448 NVIDIA V100 GPUs, Feb–Oct 2021). Licensed CC BY-NC-ND 4.0 — the data is never published or committed; only the scripts that build it.

## The Four Pillars

**1. Classical and foundation-model benchmark (RQ1).** Statistical models (ARIMA, ETS), machine learning (LightGBM), deep learning (LSTM, GRU, PatchTST, N-HiTS), and zero-shot time-series foundation models (Chronos, TimesFM) are compared under a single evaluation protocol. Special attention is paid to errors during load ramps, a known weakness of current models.

**2. Hybrid quantum-classical models (RQ2).** A hybrid QCNN and Quantum Reservoir Computing are evaluated on a simulator. The question is not whether quantum is "better", but whether it is competitive with fewer parameters or less training data. For fairness, every quantum model is compared against a parameter-matched classical control, and the role of entanglement is tested through ablation.

**3. Uncertainty quantification (RQ3).** Conformal prediction (split conformal and Adaptive Conformal Inference) wraps each forecast in a prediction interval targeting 90% coverage. Empirical coverage is reported separately for normal periods and ramp events.

**4. Grid-aware control (RQ4).** A battery controlled by Model Predictive Control (MPC) smooths the power delivered to the grid. Comparing a perfect forecast (oracle), the best model, a naive model, and no forecast translates forecasting accuracy into engineering value: ramp rate reduction and required battery capacity.

> **Guiding principle:** the success of this project does not depend on the quantum model winning. A rigorous, honest benchmark — even with a negative quantum result — is a publishable outcome.

## What this notebook does

1. Creates the project directory structure.
2. Verifies access to the MIT Supercloud data on AWS S3.

In [10]:
# ===== Create project directory structure =====
from pathlib import Path

PROJECT_DIR = Path.home() / "projects" / "datacenter-power-forecasting"

DIRS = [
    "configs",            # one YAML per experiment
    "data/raw",           # raw downloads (local only)
    "data/interim",       # intermediate files
    "data/processed",     # final modeling series
    "docs/weekly",        # weekly reports
    "docs/meetings",      # meeting notes
    "notebooks",          # EDA and figures
    "src/data",           # data pipeline
    "src/models",         # classical, deep, quantum models
    "src/eval",           # metrics and rolling-origin evaluation
    "src/control",        # battery model and MPC
    "experiments",        # outputs and result tables
]

for d in DIRS:
    (PROJECT_DIR / d).mkdir(parents=True, exist_ok=True)

# Print the resulting tree
print(PROJECT_DIR)
for p in sorted(PROJECT_DIR.rglob("*")):
    if p.is_dir():
        depth = len(p.relative_to(PROJECT_DIR).parts)
        print("    " * depth + p.name + "/")

/Users/aryaneuse/projects/datacenter-power-forecasting
    configs/
    data/
        interim/
        processed/
        raw/
    docs/
        meetings/
        weekly/
    experiments/
    notebooks/
    src/
        control/
        data/
        eval/
        models/


In [11]:
%pip install -q awscli

Note: you may need to restart the kernel to use updated packages.


!aws s3 ls --no-sign-request s3://mit-supercloud-dataset/datacenter-challenge/

In [ ]:
## Python environment

A dedicated conda environment (`dcpower`, Python 3.11) keeps project dependencies isolated from the base installation. Top-level dependencies are listed in `requirements.txt`; exact versions are pinned in `requirements.lock` so the project can be reproduced on any machine.

In [5]:
from pathlib import Path
PROJECT_DIR = Path.cwd().parent   # notebooks/ → project root
print(PROJECT_DIR)

!conda create -y -n dcpower -c conda-forge --override-channels python=3.11
!conda env list

/Users/aryaneuse/Desktop/datacenter-power-forecasting
Jupyter detected...
Retrieving notices: done
Channels:
 - conda-forge
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/miniconda3/envs/dcpower

  added / updated specs:
    - python=3.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bzip2-1.0.8                |      h4e30115_10         122 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    icu-78.3                   |  py310h579977c_2        13.4 MB  conda-forge
    libcxx-23.1.2              |       h55c6f16_0         566 KB  conda-forge
    libexpat-2.8.5             |       h6111c0a_0          68 KB  conda-forge
    libffi-3.7.0               |       h47dc5ef_1          43 KB  conda-forge
    liblzma-5.8.3              |       h8088a28_1          90 KB  conda-forge
    libpython-3.11.1

In [2]:
REQUIREMENTS = """# core
numpy
pandas
polars
duckdb
pyarrow
# plotting and stats
matplotlib
statsmodels
# data access
awscli
# deep learning and quantum
torch
pennylane
# experiment tracking
wandb
# notebooks
jupyterlab
ipykernel
"""
(PROJECT_DIR / "requirements.txt").write_text(REQUIREMENTS)
print("requirements.txt written")

requirements.txt written


In [8]:
!conda env list
!conda run --no-capture-output -n dcpower pip install --progress-bar on -r "{PROJECT_DIR}/requirements.txt"


# conda environments:
#
# *  -> active
# + -> frozen
base                 *   /opt/miniconda3
dcpower                  /opt/miniconda3/envs/dcpower



In [9]:
!conda run -n dcpower pip freeze | tee "{PROJECT_DIR}/requirements.lock"
print("\nTotal packages:", len((PROJECT_DIR / "requirements.lock").read_text().splitlines()))

annotated-types==0.8.0
anyio==4.15.1
appdirs==1.4.4
appnope==1.0.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
arrow==1.4.0
asttokens==3.0.2
async-lru==2.3.0
attrs==26.1.0
autograd==1.8.0
autoray==0.8.4
awscli==1.46.1
babel==2.18.0
beautifulsoup4==4.15.0
bleach==6.4.0
cachetools==7.2.0
certifi==2026.7.22
cffi==2.1.1
charset-normalizer==3.5.2
click==8.5.0
colorama==0.4.6
comm==0.2.3
contourpy==1.3.3
cycler==0.12.1
debugpy==1.8.22
defusedxml==0.7.1
diastatic-malt==2.15.3
docutils==0.19
duckdb==1.5.6
executing==2.2.1
fastjsonschema==2.22.2
filelock==4.0.7
fonttools==4.66.1
formulaic==1.2.2
fqdn==1.6.0
fsspec==2026.9.0
gast==0.7.0
googleapis-common-protos==1.75.5
h11==0.16.0
httpcore==1.0.9
httpx==0.28.1
idna==3.20
interface_meta==2.0.1
ipykernel==7.4.0
ipython==9.17.1
ipython_pygments_lexers==1.1.1
isoduration==20.11.0
jedi==0.20.0
Jinja2==3.1.6
jmespath==1.1.0
json5==0.15.0
jsonpointer==3.1.1
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-events==0.12.1
jupyter-lsp==

In [10]:
!conda run -n dcpower python -m ipykernel install --user --name dcpower --display-name "Python (dcpower)"

Installed kernelspec dcpower in /Users/aryaneuse/Library/Jupyter/kernels/dcpower



In [1]:
import sys, torch, pennylane as qml, polars as pl
print(sys.executable)
print("torch", torch.__version__, "| pennylane", qml.__version__, "| polars", pl.__version__)

/opt/miniconda3/envs/dcpower/bin/python
torch 2.14.1 | pennylane 0.45.1 | polars 1.44.2


In [4]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter:  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/aryaneuse/.netrc
wandb: Currently logged in as: aryan-faghirollahy (aryan-faghirollahy-uok) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import wandb, math

run = wandb.init(project="datacenter-power-forecasting", name="setup-test",
                 config={"purpose": "verify W&B logging"})
for step in range(20):
    wandb.log({"dummy_loss": math.exp(-step / 5)})
run.finish()

dummy_loss,█▇▆▅▄▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁
dummy_loss,0.02237
